<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/05_offexchange_signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Off-Exchange Signal

The fourth and final input, and the one shaped differently from the rest. The congress,
contracts, and lobbying signals read discrete *events*; this one reads trading *flow* —
off-exchange (dark-pool) volume, where large and often institutional participants trade
away from the lit market.

### A confirming signal, by design

Off-exchange flow is genuinely useful but genuinely ambiguous. A high and rising
off-exchange share shows a heavier institutional footprint in a name, yet the raw volume
does not reveal direction — the same flow can be accumulation or distribution. So this
signal reads institutional *engagement* rather than making a directional call, carries
the **lowest weight** in the composite, and earns its keep by corroborating a read the
other three signals already make.

| Feature | What it captures |
|---|---|
| `dpi` | Off-exchange short ratio (`OTC_Short / OTC_Total`) — a buying-pressure proxy |
| `volume` | Off-exchange volume (`OTC_Total`) — institutional footprint |

### What the live feed actually gives us

The off-exchange endpoint returns a **daily snapshot**, one row per ticker, with columns
`OTC_Short`, `OTC_Total`, and `DPI` (which equals `OTC_Short / OTC_Total`). Two facts
shape the design:

- **No total-market-volume column**, so "off-exchange share of all volume" is not
  computable here — the footprint feature uses the off-exchange volume level directly.
- **No per-ticker history** (one date), so there is no trend feature, and this signal
  cannot be backtested historically from the live feed. It is a current-state confirming
  overlay. The mock spans several dates only so the backtest engine has something to step
  through.

The feed covers the entire market (~$5{,}400$ tickers), most of them illiquid, so a
**liquidity floor** (`min_oe_volume`) drops thinly-traded names whose ratios are
degenerate — a micro-cap can print `DPI = 1.0` on a single sparse trade.


## Setup

Defaults to mock data; set `USE_LIVE = True` for live Quiver off-exchange data.


In [1]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data

Working in: /content/quant-equity-research


## Install the package

This notebook's code lives in the repository under `src/quant_research/`; the clone above brought it in. The cell below installs the package in editable mode and puts it on the session path.

In [2]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.2.0


## Construct the signal

We compute as of today over a $60$-day lookback, treating the most recent $20$ days as
the window whose off-exchange share is compared against the prior level. Off-exchange
data is daily, so the windows are shorter than the event-driven signals.


In [3]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.offexchange import OffExchangeSignal
import pandas as pd

client = QuiverClient(token=QUIVER_TOKEN) if USE_LIVE else QuiverClient(mock=True)
signal = OffExchangeSignal(client, lookback_days=60, min_oe_volume=1_000_000)

scores = signal.compute()
print(f"{len(scores)} tickers scored | data: {'live' if USE_LIVE else 'mock'}")

disp = scores.head(12).copy()
disp["dpi_%"] = (disp["dpi"] * 100).round(1)
disp["vol_$M"] = (disp["volume"] / 1e6).round(0)
disp[["score", "dpi_%", "vol_$M"]]

952 tickers scored | data: live


,score,dpi_%,vol_$M
ticker,,,
ADTX,100.0,47.3,200.0
BNL,95.3,85.5,1.0
ABR,94.8,84.2,5.0
ABTC,94.7,78.8,31.0
CWK,93.0,83.6,1.0
AMCR,91.6,82.6,1.0
SWKS,89.8,80.8,2.0
LPL,89.3,80.7,1.0
JBS,89.3,80.5,2.0


### Reading the output

`dpi_%` is the off-exchange short ratio (the buying-pressure proxy), and `vol_$M` is the
off-exchange volume. The `*_n` columns below are the normalized features feeding the
score.


In [4]:
comp_cols = [c for c in scores.columns if c.endswith("_n")]
scores[["score"] + comp_cols].head(8)

,score,dpi_n,volume_n
ticker,,,
ADTX,100.0,0.511,1.000
BNL,95.3,1.000,0.001
ABR,94.8,0.984,0.022
ABTC,94.7,0.914,0.151
CWK,93.0,0.975,0.002
AMCR,91.6,0.963,0.000
SWKS,89.8,0.940,0.006
LPL,89.3,0.938,0.002


### Top names, shaded by share trend

Shading by `dpi` separates names ranking high on footprint from those ranking high on
the buying-pressure proxy.


In [5]:
import plotly.express as px

top = scores.head(15).reset_index()
fig = px.bar(top, x="score", y="ticker", orientation="h",
             color="dpi_n", color_continuous_scale="Greys",
             labels={"dpi_n": "DPI (short ratio)"},
             title="Off-exchange signal — top 15 (shading = DPI)")
fig.update_layout(yaxis=dict(autorange="reversed"), height=460)
fig.show()

## Reading it honestly

- **The DPI reading is contested.** Treating a high off-exchange short ratio as net
  buying rests on the market-maker-fill argument, which is a reasonable but debated
  interpretation. The signal leans on it lightly and is weighted lowest for this reason.
- **It is a snapshot, not history.** With one date per ticker there is no momentum or
  trend, and the live feed cannot be backtested historically — this is a current-state
  overlay that confirms what the event signals surface.
- **Liquidity matters.** The market-wide universe is full of illiquid names with
  meaningless ratios; the volume floor is what keeps the ranking on real names. Raise
  `min_oe_volume` to focus on the most heavily-traded names.

This completes the four-signal set. Because off-exchange has no history to backtest, the
composite work will lean on the three event signals for the historical test and treat
off-exchange as a present-day confirming layer.


## Recap and next

All four Hobbyist signals are now built on the shared contract: congress, government
contracts, lobbying, and off-exchange. The next notebook is the payoff — a composite
scorer that blends the four into one ranking (congress $40$, contracts $30$, lobbying
$20$, off-exchange $10$), followed by a backtest of the composite through the `02`
engine to see whether blending lifts the edge above any single signal. The dashboard
then surfaces today's top composite candidates with the historical evidence behind each.
